# 5ifthNN — Google Colab L4 Training & ONNX Export Notebook

This notebook trains the **Two-Timescale Pure Neural Network (~2.0M params)** on `.5nn.gz` BedWars recordings collected by the **5ifthNN Minecraft 26.3 Fabric Mod**, supports **Behavior Cloning (BC)**, **HG-DAgger**, and **KL-Anchored PPO Self-Play**, and exports a lightweight `.onnx` file (~7.5 MB FP32, <2.5 ms CPU/tick) you can drop directly back into `.minecraft/config/5ifthnn/models/`.

In [ ]:
# 1. Mount Google Drive (saves checkpoints & datasets across Colab disconnects)
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!git clone https://github.com/5ivesaw/5ifthNN.git /content/5ifthNN || (cd /content/5ifthNN && git pull)
%cd /content/5ifthNN/python
!pip -q install -r requirements.txt

In [ ]:
# 2. Audit your uploaded .5nn.gz recordings from Google Drive BEFORE training
DATA_DIR = '/content/drive/MyDrive/5ifthNN/recordings'
CKPT_DIR = '/content/drive/MyDrive/5ifthNN/checkpoints'
!mkdir -p "$DATA_DIR" "$CKPT_DIR"
!python3 -m fifthnn.audit "$DATA_DIR"

In [ ]:
# 3. Phase 2A: Behavior Cloning (BC) Bootstrap (mirror aug + event oversampling + aux losses)
!python3 -m fifthnn.train_bc \
  --data_dir "$DATA_DIR" \
  --out "$CKPT_DIR/fifthnn_bc_v1.pt" \
  --epochs 25 \
  --batch_size 32 \
  --seq_len 128 \
  --lr 3e-4

In [ ]:
# 4. Phase 2B (Optional): KL-Anchored PPO Update on Real-Server Headless Self-Play Rollouts
ROLLOUT_DIR = '/content/drive/MyDrive/5ifthNN/selfplay_rollouts'
!mkdir -p "$ROLLOUT_DIR"
# Uncomment once you have self-play .5nn.gz episodes in ROLLOUT_DIR:
# !python3 -m fifthnn.train_ppo \
#   --rollout_dir "$ROLLOUT_DIR" \
#   --bc_anchor "$CKPT_DIR/fifthnn_bc_v1.pt" \
#   --policy "$CKPT_DIR/fifthnn_bc_v1.pt" \
#   --out "$CKPT_DIR/fifthnn_ppo_v1.pt" \
#   --kl_coef 0.08 \
#   --epochs 4

In [ ]:
# 5. Export to ONNX (FP32 + optional ultra-fast ~2MB INT8) & Verify Parity + CPU Latency
!python3 -m fifthnn.export_onnx \
  --ckpt "$CKPT_DIR/fifthnn_bc_v1.pt" \
  --out "$CKPT_DIR/fifthnn_v1.onnx" \
  --quantize_int8

print('Copy fifthnn_v1.onnx (or fifthnn_v1_int8.onnx) into .minecraft/config/5ifthnn/models/ and run /fifthnn model load fifthnn_v1.onnx')